In [ ]:
!rm -rf /content/KernelOracle

In [ ]:
%cd /content

!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/mmarcotullio/KernelOracle.git
%cd /content/KernelOracle
!git checkout main
!git status
!ls

/content
Cloning into 'KernelOracle'...
remote: Enumerating objects: 1294, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (197/197), done.
remote: Total 1294 (delta 98), reused 123 (delta 32), pack-reused 1064 (from 1)
Receiving objects: 100% (1294/1294), 62.04 MiB | 14.33 MiB/s, done.
Resolving deltas: 100% (408/408), done.
/content/KernelOracle
Already on 'main'
Your branch is up to date with 'origin/main'.
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
 data				      README.md
 LICENSE			      report.pdf
 lstm				      requirements.txt
 lstm_implementation_and_eval.ipynb   tcn
 original_code			     'TCN_Training_&_Evaluation.ipynb'


In [ ]:
!git branch

* main


In [ ]:
!pip install -q torch pandas numpy scikit-learn matplotlib tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#creating a variable for directory path
import os

DATA_DIR = "/content/drive/MyDrive/trace_data"
print(os.listdir(DATA_DIR))

['dataset_meta.json', 'test_seen.csv', 'train.csv', 'test_unseen.csv', 'all.csv', 'cpu_bound_4p.csv', 'cpu_bound_8p.csv', 'io_mixed.csv', 'sysbench_cpu_8t.csv', 'sysbench_memory_4t.csv', 'hackbench_socket_large.csv', 'hackbench_pipe_large.csv', 'data_tcn']


In [ ]:
# Setting a symbolic link
!ln -s /content/drive/MyDrive/trace_data /content/KernelOracle/trace_data
!ls -lrt /content/KernelOracle/trace_data

lrwxrwxrwx 1 root root 33 Mar 18 00:55 /content/KernelOracle/trace_data -> /content/drive/MyDrive/trace_data


In [ ]:
# %%bash
!cd /content/KernelOracle

print("Patched preprocess.py successfully.")


!rm -rf /content/KernelOracle/lstm/artifacts
!mkdir -p /content/KernelOracle/lstm/artifacts


!ls -lh /content/KernelOracle/lstm/artifacts

Patched preprocess.py successfully.
total 0


In [ ]:
%%bash
cd /content/KernelOracle

python - <<'PY'
import json
from pathlib import Path

vocab_path = Path("/content/KernelOracle/lstm/artifacts/vocab.json")
with open(vocab_path, "r") as f:
    vocab = json.load(f)

changed = False

if "UNK" not in vocab["pid_to_idx"]:
    unk_idx = len(vocab["pid_to_idx"])
    vocab["pid_to_idx"]["UNK"] = unk_idx
    vocab["idx_to_pid"][str(unk_idx)] = "UNK"
    changed = True
    print(f"Added PID UNK -> {unk_idx}")

if "UNK" not in vocab["state_to_idx"]:
    unk_idx = len(vocab["state_to_idx"])
    vocab["state_to_idx"]["UNK"] = unk_idx
    vocab["idx_to_state"][str(unk_idx)] = "UNK"
    changed = True
    print(f"Added STATE UNK -> {unk_idx}")

if changed:
    with open(vocab_path, "w") as f:
        json.dump(vocab, f)
    print("Updated vocab.json")
else:
    print("vocab.json already has UNK entries")
PY

# Removing existing npz files
rm -f /content/KernelOracle/lstm/artifacts/train.npz
rm -f /content/KernelOracle/lstm/artifacts/test_seen.npz
rm -f /content/KernelOracle/lstm/artifacts/test_unseen.npz

# Preprocessing the data
python /content/KernelOracle/lstm/scripts/preprocess.py \
  --csv /content/KernelOracle/trace_data/train.csv \
  --out /content/KernelOracle/lstm/artifacts/train.npz \
  --vocab_out /content/KernelOracle/lstm/artifacts/vocab.json \
  --seq_len 64 \
  --stride 4 \
  --use_existing_vocab

python /content/KernelOracle/lstm/scripts/preprocess.py \
  --csv /content/KernelOracle/trace_data/test_seen.csv \
  --out /content/KernelOracle/lstm/artifacts/test_seen.npz \
  --vocab_out /content/KernelOracle/lstm/artifacts/vocab.json \
  --seq_len 64 \
  --stride 4 \
  --use_existing_vocab

python /content/KernelOracle/lstm/scripts/preprocess.py \
  --csv /content/KernelOracle/trace_data/test_unseen.csv \
  --out /content/KernelOracle/lstm/artifacts/test_unseen.npz \
  --vocab_out /content/KernelOracle/lstm/artifacts/vocab.json \
  --seq_len 64 \
  --stride 4 \
  --use_existing_vocab

ls -lh /content/KernelOracle/lstm/artifacts

Saved /content/KernelOracle/lstm/artifacts/train.npz
Windows: 1812164, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: /content/KernelOracle/lstm/artifacts/vocab.json
Saved /content/KernelOracle/lstm/artifacts/test_seen.npz
Windows: 792463, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: /content/KernelOracle/lstm/artifacts/vocab.json
Saved /content/KernelOracle/lstm/artifacts/test_unseen.npz
Windows: 2172560, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: /content/KernelOracle/lstm/artifacts/vocab.json
total 121M
-rw-r--r-- 1 root root  16M Mar 18 00:57 test_seen.npz
-rw-r--r-- 1 root root  55M Mar 18 00:58 test_unseen.npz
-rw-r--r-- 1 root root  51M Mar 18 00:56 train.npz
-rw-r--r-- 1 root root 124K Mar 18 00:56 vocab.json


Traceback (most recent call last):
  File "<stdin>", line 5, in <module>
FileNotFoundError: [Errno 2] No such file or directory: '/content/KernelOracle/lstm/artifacts/vocab.json'


In [20]:
# Training LSTM
!python -m lstm.train_lstm \
  --train_npz /content/KernelOracle/lstm/artifacts/train.npz \
  --test_seen_npz /content/KernelOracle/lstm/artifacts/test_seen.npz \
  --test_unseen_npz /content/KernelOracle/lstm/artifacts/test_unseen.npz \
  --vocab /content/KernelOracle/lstm/artifacts/vocab.json \
  --epochs 5 \
  --batch_size 128 \
  --lr 0.001 \
  --hidden 256 \
  --num_layers 3 \
  --dropout 0.2 \
  --save_dir lstm/checkpoints

epoch=1 loss=5.2089 time=168.6s | seen_acc=0.0369 | unseen_acc=0.3849
Saved best checkpoint -> lstm/checkpoints/lstm_nextpid_best.pt
epoch=2 loss=4.6363 time=164.3s | seen_acc=0.0463 | unseen_acc=0.4039
Saved best checkpoint -> lstm/checkpoints/lstm_nextpid_best.pt
epoch=3 loss=4.4645 time=164.4s | seen_acc=0.0501 | unseen_acc=0.4088
Saved best checkpoint -> lstm/checkpoints/lstm_nextpid_best.pt
^C


In [21]:
# Verifying checkpoint
os.listdir("lstm/checkpoints")

['lstm_nextpid_best.pt']

In [22]:
import numpy as np

# Loading train npz file
data = np.load("lstm/artifacts/train.npz")
print(data.files)

import torch
import time
from lstm.models.lstm import LSTMConfig, LSTMNextPid
from lstm.utils.data import TraceWindowDataset, Vocab, batch_to_device
from torch.utils.data import DataLoader

#Forcing the device to run on cpu for evaluation
device = torch.device("cpu")
torch.set_num_threads(1)

print("Using device:", device)

vocab = Vocab.load("lstm/artifacts/vocab.json")

ckpt = torch.load(
    "lstm/checkpoints/lstm_nextpid_best.pt",
    map_location=device
)

cfg = LSTMConfig(**ckpt["cfg"])
model = LSTMNextPid(cfg).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

ds = TraceWindowDataset("lstm/artifacts/train.npz")
loader = DataLoader(ds, batch_size=128, shuffle=False)

batch = next(iter(loader))
batch = batch_to_device(batch, device)

pid = batch["pid"]
cont = batch["cont"]
state = batch["state"]

print("Model device:", next(model.parameters()).device)
print("pid device:", pid.device)
print("cont device:", cont.device)
print("state device:", state.device)

print("pid shape:", pid.shape)
print("cont shape:", cont.shape)
print("state shape:", state.shape)

['pid', 'cont', 'state', 'y', 'seq_len']
Using device: cpu
Model device: cpu
pid device: cpu
cont device: cpu
state device: cpu
pid shape: torch.Size([128, 64])
cont shape: torch.Size([128, 64, 2])
state shape: torch.Size([128, 64])


In [23]:
# Collecting forward latency, inference latency and throughput
warmup = 20
iters = 100

with torch.inference_mode():

    for _ in range(warmup):
        _ = model(pid, cont, state)

    start = time.perf_counter()

    for _ in range(iters):
        _ = model(pid, cont, state)

    end = time.perf_counter()

lat_ms = (end - start) / iters * 1000

batch_size = pid.shape[0]

print(f"\nAvg forward latency per batch: {lat_ms:.3f} ms (CPU)")
print(f"Latency per sample: {(lat_ms / batch_size):.6f} ms")

throughput = batch_size / (lat_ms / 1000)
print(f"Throughput: {throughput:.2f} samples/sec")


Avg forward latency per batch: 220.697 ms (CPU)
Latency per sample: 1.724196 ms
Throughput: 579.98 samples/sec
